In [1]:
# Load Packages 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from functools import reduce
from plotnine import * 
import statsmodels.formula.api as smf 
from pathlib import Path 

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True) 

In [ ]:
# Loading previously created datasets 
cas_2018_2022 = pd.read_csv('DATA_DIR/cas_refugees_by_commune_2018_2022.csv') 
merged_elections = pd.read_csv("DATA_DIR/merged_elections.csv")

In [2]:
# Loading ISTAT's Employment Dataset 

unemp_2018 = pd.read_csv("DATA_DIR/unemp_data_2018_with_regione.csv") 
unemp_2019 = pd.read_csv("DATA_DIR/unemp_data_2019_with_regione.csv") 
unemp_2021 = pd.read_csv("DATA_DIR/unemp_data_2021_with_regione.csv") 
unemp_2022 = pd.read_csv("DATA_DIR/unemp_data_2022_with_regione.csv") 

In [3]:
# Merging the datasets across the years 

unemp_2018_2019 = unemp_2018.merge(
    unemp_2019,
    on=["comune", "Regione"],
    how="left",
    suffixes=("_2018", "_2019")) 

unemp_2018_2019

,comune,employment_rate_2018,Regione,employment_rate_2019
0,Agliè,72.83,PIEMONTE,72.70
1,Airasca,71.29,PIEMONTE,71.22
2,Ala di Stura,73.41,PIEMONTE,75.00
3,Albiano d'Ivrea,69.54,PIEMONTE,71.71
4,Almese,72.66,PIEMONTE,72.68
...,...,...,...,...
7898,Villaputzu,54.35,SARDEGNA,54.70
7899,Villasalto,58.32,SARDEGNA,58.53
7900,Villasimius,61.38,SARDEGNA,62.83
7901,Villasor,53.05,SARDEGNA,53.57


In [5]:
keys = ["comune", "Regione"]

unemp_2021_renamed = unemp_2021.rename(
    columns={col: f"{col}_2021" for col in unemp_2021.columns if col not in keys})

unemp_2018_2021 = unemp_2018_2019.merge(
    unemp_2021_renamed,
    on=keys,
    how = "left") 

unemp_2018_2021

,comune,employment_rate_2018,Regione,employment_rate_2019,employment_rate_2021
0,Agliè,72.83,PIEMONTE,72.70,73.28
1,Airasca,71.29,PIEMONTE,71.22,72.09
2,Ala di Stura,73.41,PIEMONTE,75.00,68.93
3,Albiano d'Ivrea,69.54,PIEMONTE,71.71,73.25
4,Almese,72.66,PIEMONTE,72.68,74.24
...,...,...,...,...,...
7898,Villaputzu,54.35,SARDEGNA,54.70,55.96
7899,Villasalto,58.32,SARDEGNA,58.53,59.33
7900,Villasimius,61.38,SARDEGNA,62.83,61.46
7901,Villasor,53.05,SARDEGNA,53.57,57.00


In [6]:
unemp_2022_renamed = unemp_2022.rename(
    columns={col: f"{col}_2022" for col in unemp_2022.columns if col not in keys})

unemp_2018_2022 = unemp_2018_2021.merge(
    unemp_2022_renamed,
    on=keys,
    how = "left") 

unemp_2018_2022

,comune,employment_rate_2018,Regione,employment_rate_2019,employment_rate_2021,employment_rate_2022
0,Agliè,72.83,PIEMONTE,72.70,73.28,72.26
1,Airasca,71.29,PIEMONTE,71.22,72.09,70.97
2,Ala di Stura,73.41,PIEMONTE,75.00,68.93,70.11
3,Albiano d'Ivrea,69.54,PIEMONTE,71.71,73.25,71.81
4,Almese,72.66,PIEMONTE,72.68,74.24,73.44
...,...,...,...,...,...,...
7898,Villaputzu,54.35,SARDEGNA,54.70,55.96,56.78
7899,Villasalto,58.32,SARDEGNA,58.53,59.33,60.57
7900,Villasimius,61.38,SARDEGNA,62.83,61.46,63.75
7901,Villasor,53.05,SARDEGNA,53.57,57.00,58.26


In [7]:
# Calculating the mean employment rate for every commune between 2018 to 2022 

rate_emp = ["employment_rate_2018",
    "employment_rate_2019",
    "employment_rate_2021",
    "employment_rate_2022"] 

unemp_2018_2022["employment_rate_mean"] = (
    unemp_2018_2022[rate_emp].mean(axis=1))

In [9]:
# Renaming the column so that it is easier to join the datasets 
unemp_2018_2022.rename(columns={"comune": "COMUNE"}, inplace = True)

In [10]:
# Creating a data frame for regression by merging election data and refugee data

reg_data = merge_with_diagnostics(
    merged_elections,
    cas_2018_2022[["comune_id", "COMUNE", "refugees_per_1000_inhabitants_mean"]],
    on = "COMUNE",
    how = "left",
    name = "Merge elections with refugee data")

NameError: name 'merged_elections' is not defined

In [ ]:
# Standardizing column names in the regression data frame so that it is easier to merge 

reg_data["COMUNE_upper"] = (
    reg_data["COMUNE"]
    .astype(str)
    .str.strip()
    .str.upper())

unemp_2018_2022["COMUNE_upper"] = (
    unemp_2018_2022["COMUNE"]
    .astype(str)
    .str.strip()
    .str.upper()) 

In [ ]:
# Creating the already created data frame for regression with commune level employment data

reg_data_2 = reg_data.merge(
    unemp_2018_2022[["COMUNE_upper", "employment_rate_mean"]],
    on="COMUNE_upper",
    how="left") 

reg_data_2

In [ ]:
# Creating and running the linear regression with centre-left vote share change as dependent variable and refugees per 1000 inhabitants 
# as independent variable while the mean employment rate and centre-left vote share in 2018 are controls 

model_controls = smf.ols(
    "centre_left_change_22_18 ~ refugees_per_1000_inhabitants_mean + employment_rate_mean + centre_left_coalition_perc_18",
    data=reg_data_2
).fit()

print(model_controls.summary()) 

In [ ]:
# Scatterplot of Centre-Left Vote Share Change vs. Refugees per 1,000 Inhabitants

graph_plot = (ggplot(reg_data_2,
        aes(x="refugees_per_1000_inhabitants_mean",
            y="centre_left_change_22_18"))
    + geom_point(alpha=0.5, color="#264C48")
    + geom_smooth(method="lm", se=True, color="#264C48")
    + labs(title="Scatterplot of Centre-Left Vote Share Change vs. Refugees per 1,000 Inhabitants",
        x="Refugees per 1,000 Inhabitants",
        y="Centre-Left Vote Share Change, 2022–2018")
    + theme_minimal())

graph_plot 

graph_plot.save(
    OUTPUT_DIR / "centre_left_change_vs_refugees.png",
    width=8,
    height=5,
    dpi=300
)